# Publicación en el Online Feature Store

El objetivo de este script es configurar y registrar nuestra tabla de características de manufactura (`gold_machine_aggregations`) en el **Online Feature Store** (respaldado por Lakebase). 

Esto permitirá que, en producción, el modelo consulte en tiempo real el estado de degradación de una máquina (con latencia < 10ms) justo antes de predecir si la pieza será defectuosa.

**Nota de arquitectura:**
* **`gold_machine_aggregations`**: Se publica porque contiene las señales dinámicas de la máquina (vibración, desgastes temporales) necesarias para la inferencia.
* **`gold_defect_spine`**: NO se publica. Es nuestro andamio de eventos y solo se usará offline para construir el dataset de entrenamiento.

In [0]:
%pip install databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

## 1. Importación y Configuración

Importamos el cliente y definimos las rutas de nuestro catálogo. El cliente hereda automáticamente los permisos de nuestro clúster.

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

catalog = "workspace"
database = "calidad_manufactura"

gold_aggregations_table = f"{catalog}.{database}.gold_machine_aggregations"

# A single online store can host multiple feature tables.
# This is the recommended approach to reduce infrastructure costs.
# Names for the tables once published inside the online store
online_store_name = "manufacturing_quality_online_store"
online_aggregations_table = f"{catalog}.{database}.online_machine_aggregations"

# Instantiate the client using the current cluster credentials automatically
fe = FeatureEngineeringClient()

## 2. Creación del Online Feature Store

Creamos la instancia gestionada de bases de datos que servirá las características.

*(Nota: El código está comentado porque la funcionalidad de Lakebase requiere licencias de pago/Premium en Databricks, pero esta es la lógica exacta que se usaría en producción).*

In [0]:
capacity = "CU_1"  # Capacidad computacional de la instancia

# try:
#     fe.create_online_store(
#         name = online_store_name,
#         capacity = capacity
#     )
#     print(f"Online store creado: {online_store_name}")
# except Exception as e:
#     if "already exists" in str(e).lower():
#         print(f"El Online store ya existe, omitiendo creación: {online_store_name}")
#     else:
#         raise

# online_store = fe.get_online_store(name = online_store_name)
# print(f"Estado del Online store: {online_store.state}")

## 3. Publicación de Características

Publicamos la tabla usando el modo `TRIGGERED`. Elegimos este modo porque nuestra capa Gold se calcula mediante un proceso batch con ventanas deslizantes (*rolling windows*). El modo `TRIGGERED` se encarga de propagar exactamente los cambios acumulados desde la última vez que se ejecutó el pipeline, optimizando los costes de computación al no estar escuchando continuamente.

In [0]:
publish_mode = "TRIGGERED" 

# fe.publish_table(
#     online_store = online_store_name,
#     source_table_name = gold_aggregations_table,
#     online_table_name = online_aggregations_table,
#     publish_mode = publish_mode
# )
# print(f"Tabla publicada exitosamente: {gold_aggregations_table} hacia {online_aggregations_table}")